In [10]:
import sys
!{sys.executable} -m pip install python-dotenv

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [12]:
import os
import json
import time
from typing import Optional

import pandas as pd
from tqdm.auto import tqdm
from google import genai

from dotenv import load_dotenv

In [18]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY is not set")

client = genai.Client(api_key=api_key)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [105]:
SYSTEM_PROMPT = """
You are a requirements-oriented classifier for agile user stories. Your task is to classify the semantic relation between two provided user stories (Story A and Story B).

Based only on the provided story texts, you must choose exactly one label from: DUPLICATE, CONFLICT, DEPENDS_ON, NONE.

### LABEL DEFINITIONS & EXCLUSIONS

DUPLICATE
- Definition: Story A and Story B express the same core requirement intent, user need, and expected user-visible outcome. 
- Rules: Use DUPLICATE if one story is simply a more specific version of the other or adds a minor qualifier to the same core action, as long as the underlying functional requirement is identical.
- EXCLUSION (Actor Mismatch): Do NOT use DUPLICATE if the actors are different.
- EXCLUSION (Distinct Mechanisms): Do NOT use DUPLICATE if the stories describe fundamentally different technical actions, integration methods, or delivery mechanisms, even if they target the exact same entity. 

CONFLICT
- Definition: Story A and Story B are materially incompatible as stated. They cannot both be logically implemented and satisfied at the same time (e.g. if they refer to the same distinct instance but require different functionalities).
- Rules: Applies to contradictory permissions, states, approval conditions, required behaviors, access control, or runtime condition.
- EXCLUSION: Do not use CONFLICT merely because stories address different use cases or represent alternate paths.
- EXCLUSION (Warnings/Confirmations): Do NOT use CONFLICT when one story allows an action and the other story merely adds a warning, confirmation, or validation step to that same action. These are complementary, not contradictory.

DEPENDS_ON
- Definition: One story provides a capability, authorization, configuration, resource, data availability, or prior state that the other story logically requires in order to be executed or accessed.
- Rules: Use DEPENDS_ON only for clear sequential workflows where the downstream action cannot be meaningfully performed unless the upstream capability, resource, or state already exists. (e.g., modifying, updating, or deleting an entity logically requires the capability to first create, provision, or acquire that entity).
         Use DEPENDS_ON if one actor's goal relies on the safety, storage, or availability of a system resource that is provisioned or maintained by a completely different actor's goal.
- EXCLUSION: Do not use DEPENDS_ON if there is no clear requirment between stories, if the relationship is merely logical precedent or if it is simply a desire not a requirement. Do not make assumptions outside of the scope of the stories provided.
- Direction: You must determine the direction. If Story B requires Story A to happen first, Story B depends on Story A.

NONE
- Definition: None of the above relations are sufficiently supported by the text alone.
- Rules: Use NONE for stories that are merely topically related, exist in the same feature area, or share keywords but lack a strict dependency, duplication, or conflict. Prefer NONE over speculative relation assignments.

### DECISION RULES
1. Grounding: Base the decision ONLY on the content of Story A and Story B. Do not infer unstated background assumptions.
2. Step-by-Step Logic: First, evaluate the actors to ensure they match or interact functionally. Second, evaluate the core actions. Third, check at first whether the stories have a CONFLICT relation, if not, check for DUPLICATE, and if not only then check DEPENDS_ON relationship using the rules above.
3. Output formatting: If the label is DEPENDS_ON, set "source_story" to the story that depends on the other, and "target_story" to the story it requires. For all other labels, set both direction fields to null.

### EXAMPLES

Example 1:
Story A: As a Customer, I want to add items to my digital shopping cart, so that I can purchase multiple books at once.
Story B: As a Customer, I want to change the quantity of an item already in my shopping cart, so that I can buy multiple copies of the same book.
Output:
{
  "reason": "Story B involves modifying the quantity of items already in a cart. Logically, a downstream modification action requires the upstream capability to acquire/add items first (Story A). Therefore, B requires A as a workflow prerequisite.",
  "relation_type": "DEPENDS_ON",
  "source_story": "B",
  "target_story": "A"
}

Example 2:
Story A: As a Store Manager, I want to view a daily sales report, so that I can track overall revenue.
Story B: As a Store Manager, I want to view a daily sales report filtered by book genre, so that I can see which categories perform best.
Output:
{
  "reason": "Both stories involve the exact same actor viewing the exact same core report. Story B is simply a slightly refined version (adding a filter qualifier) of the core requirement in Story A. One subsumes the other.",
  "relation_type": "DUPLICATE",
  "source_story": null,
  "target_story": null
}

Example 3:
Story A: As a Warehouse Worker, I want an alert when stock drops below 10 units, so I can reorder.
Story B: As a Customer, I want to see an 'Out of Stock' badge on unavailable books, so I don't try to buy them.
Output:
{
  "reason": "The actors represent completely distinct roles with different viewpoints (Internal Worker vs External Customer). While both relate to the topic of inventory, one is an internal backend alert and the other is an external UI badge. Neither logically requires the other to function.",
  "relation_type": "NONE",
  "source_story": null,
  "target_story": null
}

Example 4:
Story A: As a System Admin, I want user passwords to expire every 30 days, to ensure security.
Story B: As a System Admin, I want user passwords to never expire, to reduce support tickets.
Output:
{
  "reason": "The rules established in both stories are materially incompatible. The system cannot simultaneously force passwords to expire every 30 days and allow them to never expire.",
  "relation_type": "CONFLICT",
  "source_story": null,
  "target_story": null
}

### OUTPUT FORMAT
Return EXACTLY ONE valid JSON object. Do not include markdown formatting or outside text.
{
  "reason": "1-2 sentences grounded only in the text, evaluating actors and core actions.",
  "relation_type": "DUPLICATE|CONFLICT|DEPENDS_ON|NONE",
  "source_story": "A|B|null",
  "target_story": "A|B|null"
}
""".strip()

In [107]:
def classify_story_pair(
    story_a: str,
    story_b: str,
    story_a_id: Optional[str] = None,
    story_b_id: Optional[str] = None,
    model_name: str = 'gemini-3-pro-preview'
):
    user_prompt = f"""
Story A ID: {story_a_id}
Story A:
{story_a}

Story B ID: {story_b_id}
Story B:
{story_b}

Return only valid JSON.
""".strip()

    full_prompt = f"{SYSTEM_PROMPT}\n\n{user_prompt}"

    response = client.models.generate_content(
        model=model_name,
        contents=full_prompt,
    )

    raw_text = response.text.strip()

    try:
        parsed = json.loads(raw_text)
    except json.JSONDecodeError:
        import re
        match = re.search(r"\{.*\}", raw_text, re.DOTALL)
        if not match:
            raise ValueError(f"Could not parse JSON from model output:\n{raw_text}")
        parsed = json.loads(match.group(0))

    relation_type = parsed.get("relation_type", "").strip().upper()
    reason = parsed.get("reason", "").strip()

    allowed = {"DUPLICATE", "CONFLICT", "DEPENDS_ON", "NONE"}
    if relation_type not in allowed:
        raise ValueError(f"Invalid label returned: {relation_type}")

    return {
        "story_a_id": story_a_id,
        "story_b_id": story_b_id,
        "story_a": story_a,
        "story_b": story_b,
        "relation_type": relation_type,
        "reason": reason,
        "raw_response": raw_text,
    }

In [109]:
def classify_row(row):
    result = classify_story_pair(
        story_a=row["story_1_text"],
        story_b=row["story_2_text"],
        story_a_id=row["pair_id_1"],
        story_b_id=row["pair_id_2"],
        model_name="gemini-3-pro-preview"
    )
    return pd.Series({
        "predicted_relation": result["relation_type"],
        "llm_reason": result["reason"],
        "raw_response": result.get("raw_response", "")
    })
    

In [119]:
# UPLOAD THE PREPROCESSED DATASET
candidate_pairs_final = pd.read_csv(r'pred_res\neo4j\g14_inter_story_for_llm.csv')
candidate_pairs_final

,pair_id_1,pair_id_2,transformation,story_1_text,story_2_text,predicted_relation,llm_reason,response_time_sec,true_label,data_source,raw_response,source_story,target_story
0,0,1,NaN,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a dataset, s...",CONFLICT,Both stories describe the act of publishing a ...,0.839088,CONFLICT,natural,"{\n ""relation_type"": ""CONFLICT"",\n ""source_s...",NaN,NaN
1,0,21,NaN,"As a Publisher, I want to publish a dataset, s...","As a Consumer, I want to view a data package o...",NONE,The stories involve different actors (Publishe...,0.720764,NONE,natural,"{\n ""relation_type"": ""NONE"",\n ""source_story...",NaN,NaN
2,0,22,NaN,"As a Publisher, I want to publish a dataset, s...","As a publisher, I want to show the world how m...",CONFLICT,Story A describes a requirement for restricted...,1.137041,NONE,natural,"{\n ""relation_type"": ""NONE"",\n ""source_story...",NaN,NaN
3,0,46,NaN,"As a Publisher, I want to publish a dataset, s...","As a Consumer, I want to view a Datapackage at...",NONE,The actors are different (Publisher vs. Consum...,0.968363,NONE,natural,"{\n ""relation_type"": ""NONE"",\n ""source_story...",NaN,NaN
4,0,58,NaN,"As a Publisher, I want to publish a dataset, s...","As an owner, I want to view all the people in ...",NONE,The stories involve different actors (Publishe...,0.706517,NONE,natural,"{\n ""relation_type"": ""NONE"",\n ""source_story...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,59,60,NaN,"As an owner, I want to make a user an owner, s...","As an owner, I want to remove a user as an own...",NONE,Story A and Story B describe opposite administ...,0.652525,NONE,natural,"{\n ""relation_type"": ""NONE"",\n ""source_story...",NaN,NaN
67,59,63,NaN,"As an owner, I want to make a user an owner, s...","As an Admin, I want to have a pricing plan and...",NONE,The stories involve different actors (Owner vs...,0.791965,NONE,natural,"{\n ""relation_type"": ""NONE"",\n ""source_story...",NaN,NaN
68,63,64,NaN,"As an Admin, I want to have a pricing plan and...","As a Publisher, I want to know if this site ha...",NONE,The stories involve different actors (Admin vs...,0.789064,DEPENDS_ON,natural,"{\n ""relation_type"": ""NONE"",\n ""source_story...",NaN,NaN
69,63,65,NaN,"As an Admin, I want to have a pricing plan and...","As a Publisher, I want to sign up for a given ...",DEPENDS_ON,Story B describes a publisher signing up for a...,0.821794,DEPENDS_ON,natural,"{\n ""relation_type"": ""DEPENDS_ON"",\n ""source...",B,A


In [121]:
test_df = candidate_pairs_final.head(3).copy()
test_preds = test_df.apply(classify_row, axis=1)
test_result = pd.concat([test_df, test_preds], axis=1)
test_result

,pair_id_1,pair_id_2,transformation,story_1_text,story_2_text,predicted_relation,llm_reason,response_time_sec,true_label,data_source,raw_response,source_story,target_story,predicted_relation,llm_reason,raw_response
0,0,1,NaN,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a dataset, s...",CONFLICT,Both stories describe the act of publishing a ...,0.839088,CONFLICT,natural,"{\n ""relation_type"": ""CONFLICT"",\n ""source_s...",NaN,NaN,NONE,Both stories involve the same actor publishing...,"{\n ""reason"": ""Both stories involve the same ..."
1,0,21,NaN,"As a Publisher, I want to publish a dataset, s...","As a Consumer, I want to view a data package o...",NONE,The stories involve different actors (Publishe...,0.720764,NONE,natural,"{\n ""relation_type"": ""NONE"",\n ""source_story...",NaN,NaN,DEPENDS_ON,Story B involves a Consumer viewing a dataset ...,"{\n ""reason"": ""Story B involves a Consumer vi..."
2,0,22,NaN,"As a Publisher, I want to publish a dataset, s...","As a publisher, I want to show the world how m...",CONFLICT,Story A describes a requirement for restricted...,1.137041,NONE,natural,"{\n ""relation_type"": ""NONE"",\n ""source_story...",NaN,NaN,DEPENDS_ON,Story B involves showing 'published data' to t...,"{\n ""reason"": ""Story B involves showing 'publ..."


In [122]:
def run_cross_story_experiment(
    df,
    n_rows=None,
    model_name="gemini-3-pro-preview",
    save_every=None
):
    """
    Run cross-story classification on the first n_rows of df.
    
    Parameters
    ----------
    df : pandas.DataFrame
        Must contain:
        - pair_id_1
        - pair_id_2
        - story_1_text
        - story_2_text
    n_rows : int or None
        If set, only run on the first n_rows rows.
        If None, run on the full dataframe.
    model_name : str
        Gemini model name.
    save_every : int or None
        If set, saves intermediate CSV every N rows.
    """
    
    if n_rows is not None:
        run_df = df.head(n_rows).copy().reset_index(drop=True)
    else:
        run_df = df.copy().reset_index(drop=True)

    results = []
    run_start = time.perf_counter()

    for idx, row in tqdm(run_df.iterrows(), total=len(run_df), desc=f"Running {len(run_df)} pairs"):
        row_start = time.perf_counter()
        # print('a', row["story_1_text"])
        # print('b',row["story_2_text"])
        # print('c', row["pair_id_1"])
        # print('d', row["pair_id_2"])
        try:
            result = classify_story_pair(
                story_a=row["story_1_text"],
                story_b=row["story_2_text"],
                story_a_id=row["pair_id_1"],
                story_b_id=row["pair_id_2"],
                model_name=model_name
            )

            row_end = time.perf_counter()
            row_seconds = row_end - row_start

            results.append({
                "row_index": idx,
                "pair_id_1": row["pair_id_1"],
                "pair_id_2": row["pair_id_2"],
                "candidate_source": row.get("candidate_source", None),
                "score": row.get("score", None),
                "predicted_relation": result.get("relation_type", None),
                "llm_reason": result.get("reason", None),
                "raw_response": result.get("raw_response", None),
                "response_time_sec": row_seconds
            })

            print(
                f"Row {idx} done | pair ({row['pair_id_1']}, {row['pair_id_2']}) "
                f"| label={result.get('relation_type', 'NA')} "
                f"| time={row_seconds:.2f}s"
            )

        except Exception as e:
            row_end = time.perf_counter()
            row_seconds = row_end - row_start

            results.append({
                "row_index": idx,
                "pair_id_1": row["pair_id_1"],
                "pair_id_2": row["pair_id_2"],
                "candidate_source": row.get("candidate_source", None),
                "score": row.get("score", None),
                "predicted_relation": "ERROR",
                "llm_reason": str(e),
                "raw_response": None,
                "response_time_sec": row_seconds
            })

            print(
                f"Row {idx} ERROR | pair ({row['pair_id_1']}, {row['pair_id_2']}) "
                f"| time={row_seconds:.2f}s | error={e}"
            )

        if save_every is not None and (idx + 1) % save_every == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(f"temp_results_{idx+1}_rows.csv", index=False)

    run_end = time.perf_counter()
    total_seconds = run_end - run_start

    results_df = pd.DataFrame(results)

    summary = {
        "n_rows": len(run_df),
        "total_time_sec": total_seconds,
        "avg_time_per_row_sec": results_df["response_time_sec"].mean() if len(results_df) > 0 else None,
        "min_time_sec": results_df["response_time_sec"].min() if len(results_df) > 0 else None,
        "max_time_sec": results_df["response_time_sec"].max() if len(results_df) > 0 else None,
    }

    print("\n===== RUN SUMMARY =====")
    print(f"Rows processed: {summary['n_rows']}")
    print(f"Total time: {summary['total_time_sec']:.2f} sec")
    print(f"Average / row: {summary['avg_time_per_row_sec']:.2f} sec")
    print(f"Min row time: {summary['min_time_sec']:.2f} sec")
    print(f"Max row time: {summary['max_time_sec']:.2f} sec")

    return results_df, summary

In [59]:
# results_3, summary_3 = run_cross_story_experiment(candidate_pairs_final, n_rows=3, model_name="gemini-3-pro-preview")
# results_5, summary_5 = run_cross_story_experiment(candidate_pairs_final, n_rows=5, model_name="gemini-3-pro-preview")
# results_7, summary_7 = run_cross_story_experiment(candidate_pairs_final, n_rows=7, model_name="gemini-3-pro-preview")

In [125]:
results_full, summary_full = run_cross_story_experiment(candidate_pairs_final, n_rows=None, model_name="gemini-3.1-flash-lite-preview")

Running 71 pairs:   0%|          | 0/71 [00:00<?, ?it/s]

Row 0 done | pair (0, 1) | label=CONFLICT | time=1.24s
Row 1 done | pair (0, 21) | label=NONE | time=0.82s
Row 2 done | pair (0, 22) | label=CONFLICT | time=0.81s
Row 3 done | pair (0, 46) | label=NONE | time=0.96s
Row 4 done | pair (0, 58) | label=NONE | time=0.88s
Row 5 done | pair (1, 21) | label=DEPENDS_ON | time=0.96s
Row 6 done | pair (1, 22) | label=NONE | time=0.88s
Row 7 done | pair (1, 24) | label=DEPENDS_ON | time=0.81s
Row 8 done | pair (2, 8) | label=DEPENDS_ON | time=1.02s
Row 9 done | pair (2, 11) | label=NONE | time=1.03s
Row 10 done | pair (2, 39) | label=DEPENDS_ON | time=0.99s
Row 11 done | pair (2, 51) | label=NONE | time=0.84s
Row 12 done | pair (2, 63) | label=NONE | time=0.82s
Row 13 done | pair (2, 65) | label=DUPLICATE | time=1.02s
Row 14 done | pair (8, 11) | label=DEPENDS_ON | time=0.75s
Row 15 done | pair (8, 39) | label=NONE | time=0.84s
Row 16 done | pair (9, 18) | label=DEPENDS_ON | time=0.91s
Row 17 done | pair (9, 19) | label=NONE | time=0.96s
Row 18 do

In [31]:
# timing_comparison = pd.DataFrame([
#     summary_3,
#     summary_5,
#     summary_7,
#     summary_full
# ])

# timing_comparison

In [127]:
results_full[results_full['predicted_relation']!='NONE']['predicted_relation'].value_counts()

predicted_relation
DEPENDS_ON    13
DUPLICATE      7
CONFLICT       2
Name: count, dtype: int64

In [62]:
results_full[results_full['predicted_relation']!='NONE']['predicted_relation'].value_counts()

predicted_relation
DEPENDS_ON    46
DUPLICATE     23
CONFLICT       4
Name: count, dtype: int64

In [33]:
results_full

,row_index,pair_id_1,pair_id_2,candidate_source,score,predicted_relation,llm_reason,raw_response,response_time_sec
0,0,3,3,None,None,DUPLICATE,Both stories describe the exact same actor (Vi...,"{\n ""reason"": ""Both stories describe the exac...",1.054808
1,1,3,3,None,None,CONFLICT,Story A and Story B describe diametrically opp...,"{\n ""reason"": ""Story A and Story B describe d...",0.967175
2,2,3,3,None,None,NONE,"The stories describe different, mutually exclu...","{\n ""reason"": ""The stories describe different...",1.458665
3,3,3,3,None,None,DEPENDS_ON,The stories describe the same feature from dif...,"{\n ""reason"": ""The stories describe the same ...",0.990334
4,4,4,4,None,None,DUPLICATE,"Both stories describe the same actor, intent, ...","{\n ""reason"": ""Both stories describe the same...",1.038012
...,...,...,...,...,...,...,...,...,...
111,111,16,16,None,None,DUPLICATE,Both stories describe the act of publishing a ...,"{\n ""reason"": ""Both stories describe the act ...",0.965389
112,112,63,63,None,None,DUPLICATE,Both stories involve the same actor and the sa...,"{\n ""reason"": ""Both stories involve the same ...",0.846886
113,113,19,19,None,None,CONFLICT,Story A requires that undeleting a data packag...,"{\n ""reason"": ""Story A requires that undeleti...",1.180195
114,114,22,22,None,None,NONE,Story A and Story B represent fundamentally di...,"{\n ""reason"": ""Story A and Story B represent ...",0.919954


In [23]:
df2 = candidate_pairs_final.merge(
    results_full[
        ["pair_id_1", "pair_id_2", "predicted_relation", "llm_reason", "raw_response", "response_time_sec"]
    ],
    on=["pair_id_1", "pair_id_2"],
    how="left"
)

In [91]:
df_tmp1 = df2[df2['predicted_relation'] != 'NONE']
for i, row in df_tmp1.iterrows():
    print(f"Index: {i}")
    print(f"Predicted relation: {row['predicted_relation']}")
    print(f"Story id: {row['pair_id_1']} ")
    print(row["story_1_text"])
    print()
    print(f"Story id: {row['pair_id_2']} ")
    print(row["story_2_text"])
    print()
    print("LLM reason:")
    print(row["llm_reason"])
    print("\n" + "=" * 80 + "\n")

Index: 0
Predicted relation: CONFLICT
Story id: 0 
As a Publisher, I want to publish a dataset, so that I can view just the dataset with a few people.

Story id: 1 
As a Publisher, I want to publish a dataset, so that I can share the dataset publicly with everyone.

LLM reason:
Both stories involve the same actor performing the same core action of publishing a dataset, but they define mutually exclusive access levels (restricted sharing vs. public sharing). Because the system cannot simultaneously restrict access to a few people and make the dataset available to everyone, these requirements are in conflict.


Index: 1
Predicted relation: DEPENDS_ON
Story id: 0 
As a Publisher, I want to publish a dataset, so that I can view just the dataset with a few people.

Story id: 7 
As a Publisher, I want to configure my client, so that I can start publishing data packages.

LLM reason:
Publishing a dataset (Story A) requires the prerequisite configuration of the client (Story B) to establish th

### SAVING

In [79]:
temp_df = pd.DataFrame(df2)
temp_df.to_csv(f"pred_res\g14-datahub_FINAL_PROMPT_transformations_new2.csv", index=False)

<>:2: SyntaxWarning: invalid escape sequence '\g'
<>:2: SyntaxWarning: invalid escape sequence '\g'
C:\Users\havagyan\AppData\Local\Temp\ipykernel_19464\766154093.py:2: SyntaxWarning: invalid escape sequence '\g'
  temp_df.to_csv(f"pred_res\g14-datahub_FINAL_PROMPT_transformations_new2.csv", index=False)


In [131]:
results_full.to_csv(r"pred_res\neo4j\g14_inter_story_for_llm_predicted.csv", index=False)

In [85]:
results_full

,row_index,pair_id_1,pair_id_2,candidate_source,score,predicted_relation,llm_reason,raw_response,response_time_sec
0,0,3,3,None,None,CONFLICT,The stories describe fundamentally incompatibl...,"{\n ""reason"": ""The stories describe fundament...",1.036399
1,1,4,4,None,None,CONFLICT,Story A expresses a user requirement for guida...,"{\n ""reason"": ""Story A expresses a user requi...",0.864188
2,2,5,5,None,None,CONFLICT,Story A defines an action (inviting a user) th...,"{\n ""reason"": ""Story A defines an action (inv...",0.876515
3,3,25,25,None,None,CONFLICT,The stories involve different actors (Consumer...,"{\n ""reason"": ""The stories involve different ...",0.796284
4,4,26,26,None,None,CONFLICT,Story A and Story B describe contradictory acc...,"{\n ""reason"": ""Story A and Story B describe c...",0.904144
5,5,30,30,None,None,CONFLICT,Story A defines a requirement for a Consumer t...,"{\n ""reason"": ""Story A defines a requirement ...",0.787136
6,6,31,31,None,None,CONFLICT,Story A seeks to enable the use of a data pack...,"{\n ""reason"": ""Story A seeks to enable the us...",0.891374
7,7,32,32,None,None,CONFLICT,Story A defines a capability for a Consumer to...,"{\n ""reason"": ""Story A defines a capability f...",0.894879
8,8,35,35,None,None,CONFLICT,The stories describe materially incompatible p...,"{\n ""reason"": ""The stories describe materiall...",1.004755
9,9,36,36,None,None,CONFLICT,Story A describes a desired functional capabil...,"{\n ""reason"": ""Story A describes a desired fu...",0.723666


### Exploring the final dataset

In [287]:
import pandas as pd
dff = pd.read_csv(f'pred_res\g14-datahub_FINAL_PROMPT.csv')
dff

<>:2: SyntaxWarning: invalid escape sequence '\g'
<>:2: SyntaxWarning: invalid escape sequence '\g'
C:\Users\havagyan\AppData\Local\Temp\ipykernel_18968\1469134517.py:2: SyntaxWarning: invalid escape sequence '\g'
  dff = pd.read_csv(f'pred_res\g14-datahub_FINAL_PROMPT.csv')


,pair_id_1,pair_id_2,story_1_text,story_2_text,candidate_source,score,predicted_relation,llm_reason,raw_response,response_time_sec
0,0,1,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a dataset, s...",cosine,0.900064,CONFLICT,Story A specifies publishing a dataset for a l...,"{\n ""relation_type"": ""CONFLICT"",\n ""source_s...",1.066698
1,0,7,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to configure my client,...",cosine,0.620692,NONE,Story A describes the intent to restrict visib...,"{\n ""relation_type"": ""NONE"",\n ""source_story...",1.220146
2,0,16,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a data packa...",cosine,0.689316,NONE,Story A focuses on restricting access to a dat...,"{\n ""relation_type"": ""NONE"",\n ""source_story...",1.128119
3,0,17,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to create a data packag...",cosine,0.651922,NONE,Story A describes restricting dataset visibili...,"{\n ""relation_type"": ""NONE"",\n ""source_story...",0.895258
4,0,21,"As a Publisher, I want to publish a dataset, s...","As a Consumer, I want to view a data package o...",bm25_enhanced,21.323694,NONE,The stories involve different user roles (Publ...,"{\n ""relation_type"": ""NONE"",\n ""source_story...",1.158858
...,...,...,...,...,...,...,...,...,...,...
349,61,62,"As an Admin, I want to set key configuration p...","As an Admin, I want to see key metrics about u...","bm25_enhanced, cosine",17.204144,NONE,Story A focuses on administrative configuratio...,"{\n ""relation_type"": ""NONE"",\n ""source_story...",1.351542
350,62,63,"As an Admin, I want to see key metrics about u...","As an Admin, I want to have a pricing plan and...","bm25_enhanced, cosine",16.570193,NONE,Story A focuses on administrative analytics an...,"{\n ""relation_type"": ""NONE"",\n ""source_story...",2.060128
351,63,64,"As an Admin, I want to have a pricing plan and...","As a Publisher, I want to know if this site ha...",cosine,0.527958,NONE,Story A focuses on the implementation of a bil...,"{\n ""relation_type"": ""NONE"",\n ""source_story...",1.347895
352,63,65,"As an Admin, I want to have a pricing plan and...","As a Publisher, I want to sign up for a given ...",cosine,0.618391,DEPENDS_ON,Story B depends on the pricing plan and billin...,"{\n ""relation_type"": ""DEPENDS_ON"",\n ""source...",1.144083


In [297]:

tmp = pd.concat([
    dff[["pair_id_1", "predicted_relation"]]
        .rename(columns={"pair_id_1": "pair_id"}),
    
    dff[["pair_id_2", "predicted_relation"]]
        .rename(columns={"pair_id_2": "pair_id"})
], ignore_index=True)

final_list = (
    tmp.groupby("pair_id")["predicted_relation"]
       .apply(lambda s: s.eq("NONE").all())
)

final_list = final_list[final_list].index.tolist()

print(sorted(final_list))
print(len(sorted(final_list)))
# [3, 4, 5, 6, 8, 9, 13, 22, 25, 26, 30, 31, 32, 33, 34, 35, 37, 38, 43, 45, 46, 48, 49, 54, 57, 60, 61, 62]


[3, 4, 5, 6, 8, 9, 13, 22, 25, 26, 30, 31, 32, 33, 34, 35, 37, 38, 43, 45, 46, 48, 49, 54, 57, 60, 61, 62]
28


In [309]:
ids = final_list

mask_1 = dff["pair_id_1"].isin(ids)
mask_2 = dff["pair_id_2"].isin(ids)

stories_1 = dff.loc[mask_1, ["pair_id_1", "story_1_text"]].rename(
    columns={"pair_id_1": "pair_id", "story_1_text": "story_text"}
)

stories_2 = dff.loc[mask_2, ["pair_id_2", "story_2_text"]].rename(
    columns={"pair_id_2": "pair_id", "story_2_text": "story_text"}
)

stories = (
    pd.concat([stories_1, stories_2], ignore_index=True)
    .drop_duplicates()
    .sort_values("pair_id")
)

stories['story_text'].tolist()

["As a Visitor, I want to sign up via github or google, so that that I don't have to enter lots of information and remember my password for yet another website.",
 'As a Publisher, I want to know what do next after signing up, so that that I can get going quickly.',
 'As an Admin, I want to invite someone to join the platform, so that that they can start contributing or using data.',
 'As a Publisher, I want to import my data package into the registry, so that my data has a permanent online home to access.',
 'As a Publisher, I want to use a publish command to update a data package that is already in the registry, so that it appears there.',
 'As a Publisher, I want to unpublish a data package, so that it is no longer visible to anyone.',
 "As a Consumer, I want to know that the data I am downloading is good and can be relied on, so that that I don't have to check it myself or run into annoying bugs later on.",
 "As a publisher, I want to show the world how my published data is, so tha